# ecommerce_analytics_demo — Pipeline Walkthrough

**Multi-table enrichment, margin analysis, and temporal trends.**

| Step | Layer | What happens |
|------|-------|--------------|
| 1 | Bronze | Seed 3 CSVs → `ORDERS`, `PRODUCTS`, `CUSTOMERS` Parquet |
| 2 | Silver (phase 1) | Rename + cast each table |
| 2 | Silver (phase 2) | UDF joins all three → `order_lines_enriched` |
| 3 | Gold | 3 aggregations (revenue by category, top customers, monthly trend) |

Run cells top to bottom.

## Setup — navigate to example root

In [ ]:
import os
from pathlib import Path

# Two levels up: ipynb/ → ecommerce/ → ecommerce_analytics_demo/
example_root = Path(os.getcwd()).parent.parent
os.chdir(example_root)
print(f'Working directory: {os.getcwd()}')

## Step 1 — Bronze: seed from 3 source CSVs

Reads `data/source/{orders,products,customers}.csv` and writes uppercase-column Parquet files to `data/bronze/`.
Uppercasing matches the column-name convention of a real dlt `sql_database` ingestion,
so the silver rename transforms work identically in both demo and production.

In [ ]:
import polars as pl

src  = Path('data/source')
dest = Path('data/bronze')
dest.mkdir(parents=True, exist_ok=True)

for name in ('orders', 'products', 'customers'):
    df  = pl.read_csv(src / f'{name}.csv')
    df  = df.rename({col: col.upper() for col in df.columns})
    out = dest / f'{name.upper()}.parquet'
    df.write_parquet(out)
    print(f'✅  {len(df):>3} rows → {out}')

In [ ]:
print('── ORDERS (first 4 rows) ──')
print(pl.read_parquet('data/bronze/ORDERS.parquet').head(4))
print('\n── PRODUCTS ──')
print(pl.read_parquet('data/bronze/PRODUCTS.parquet'))
print('\n── CUSTOMERS ──')
print(pl.read_parquet('data/bronze/CUSTOMERS.parquet'))

## Step 2 — Silver: rename, cast, and join

**Phase 1** — `backend/silver.yaml` renames and casts each table individually.

**Phase 2** — `build_order_lines_enriched()` (UDF) joins the 3 silver tables:
- Adds `line_revenue = qty × unit_price`
- Adds `line_cost = qty × unit_cost`
- Adds `margin_amount = line_revenue − line_cost`

In [ ]:
!medallion run ecommerce --layer silver

In [ ]:
enriched = pl.read_parquet('data/silver/order_lines_enriched.parquet')
print(f'Enriched table: {enriched.shape[0]} rows × {enriched.shape[1]} cols')
display_cols = ['order_id', 'name', 'product_name', 'category', 'qty',
                'unit_price', 'line_revenue', 'margin_amount']
print(enriched.select(display_cols).sort('order_id').head(6))

## Step 3 — Gold: 3 aggregations

All aggregations run on `order_lines_enriched.parquet`:

| Output | Group by | Metrics |
|--------|----------|---------|
| `revenue_by_category` | category | total_revenue, total_margin, num_orders |
| `top_customers` | customer_id, name, region, tier | total_spent, num_orders |
| `monthly_summary` | order_month (via pre_agg_udf) | monthly_revenue, num_orders |

In [ ]:
!medallion run ecommerce --layer gold

In [ ]:
gold = Path('data/gold/ecommerce')

print('── Revenue by Category ──')
cat = pl.read_parquet(gold / 'revenue_by_category.parquet').sort('total_revenue', descending=True)
cat = cat.with_columns(
    (pl.col('total_margin') / pl.col('total_revenue') * 100).round(1).alias('margin_pct')
)
print(cat)

In [ ]:
print('── Top Customers (by spend) ──')
cust = pl.read_parquet(gold / 'top_customers.parquet').sort('total_spent', descending=True)
print(cust)

gold_tier_rev = cust.filter(pl.col('tier') == 'gold')['total_spent'].sum()
total_rev     = cust['total_spent'].sum()
print(f'\nGold-tier revenue share: {gold_tier_rev / total_rev * 100:.1f}%')

In [ ]:
print('── Monthly Revenue Trend ──')
monthly = pl.read_parquet(gold / 'monthly_summary.parquet').sort('order_month')
print(monthly)

peak = monthly.sort('monthly_revenue', descending=True).row(0, named=True)
print(f"\nPeak month: {peak['order_month']}  (${peak['monthly_revenue']:,.2f})")

## Things to Try

- **Add margin_pct**: compute `total_margin / total_revenue` in `add_metrics()` and add `agg: mean` to `backend/gold.yaml`
- **Filter cancelled orders**: add a `filter` transform in `backend/silver.yaml` for `status != 'cancelled'`
- **Revenue by region**: add a new aggregation to `backend/gold.yaml` grouping `top_customers` by `region`